In [1]:
import pandas as pd
df_tr = pd.read_csv('../data/train.csv')
df_te = pd.read_csv('../data/test.csv')
df_tr

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [2]:
df_tr.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


# EDA

In [3]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# set visual theme
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8,5)

In [5]:
#1. Summary statistics for numerical variables
print("--- Numerical Data Summary ---")
display(df_tr.describe().T)

# 2. Precise missing values summary
missing_count = df_tr.isnull().sum()
missing_pct = (missing_count/ len(df_tr))*100
missing_df = pd.DataFrame({'Missing Values': missing_count, 'Percantage (%)':missing_pct})
print("\n--- Missing Data Breakdown ---")
display(missing_df[missing_df['Missing Values']>0])

--- Numerical Data Summary ---


,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292



--- Missing Data Breakdown ---


,Missing Values,Percantage (%)
Age,177,19.865320
Cabin,687,77.104377
Embarked,2,0.224467


In [6]:
X = df_tr.drop(
    columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Embarked', 'Survived']
).copy()
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

# 3. Fill missing Age values
X['Age'] = X['Age'].fillna(X['Age'].median())
y = df_tr['Survived']

In [7]:
X_test = df_te.drop(
    columns = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Embarked']
).copy()

X_test['Sex'] = X_test['Sex'].map({'male':0, 'female':1})

X_test['Age'] = X_test['Age'].fillna(X_test['Age'].median())
X_test['Fare'] = X_test['Fare'].fillna(X_test['Fare'].median())

# Machine Learning

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

model_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
])

# 3. Evaluate with 5-Fold Cross-Validation (Leak-Free)
cv_scores = cross_val_score(model_pipeline, X, y, cv=5)

# 4. Train Pipeline on Full Training Set
model_pipeline.fit(X, y)
train_acc = model_pipeline.score(X, y) * 100

# 5. Output Diagnostic Scores
print(f'Train Accuracy: {train_acc:.2f}%')
print(
    f'Clean 5-Fold CV Accuracy: {cv_scores.mean() * 100:.2f}% (+/-'
    f' {cv_scores.std() * 100:.2f}%)'
)

Train Accuracy: 85.52%
Clean 5-Fold CV Accuracy: 81.60% (+/- 2.25%)


# Submission

In [9]:
predictions = model_pipeline.predict(X_test)
submission = pd.DataFrame(
    {'PassengerId':df_te['PassengerId'], 'Survived': predictions}
)
submission.to_csv('../data/submission.csv', index=False)
print("Submission saved successfuly to ../data/submission.csv")

Submission saved successfuly to ../data/submission.csv
